Demo: One-class SVM with Probability.



In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import expit
from scipy.stats import gamma
from sklearn.datasets import load_breast_cancer
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

# seed giống nhau thì mới chạy ra kết quả giống nhau được, k bị ảnh hưởng bởi random
rng = np.random.default_rng(42)

1. Chuẩn bị dữ liệu.

Dùng tập dữ liệu breast cancer (bình thường  = khỏe mạnh/lành tính, bất thường thì là u ác tính)

In [ ]:

bc = load_breast_cancer()
Xall, yall = bc.data, bc.target

# find index normal vs anormal (binh thuong vs bat thuong)
normal_idx = np.where(yall == 1)[0]    # lanh tinh
anomaly_idx = np.where(yall == 0)[0]   # ac tinh
rng.shuffle(normal_idx)

# train and test set (60% train)
cut = int(len(normal_idx) * 0.6)
fit_idx = normal_idx[:cut]
held_normal = normal_idx[cut:]

# chuan hoa dac trung: scaler chi fit tren tap train
sc = StandardScaler().fit(Xall[fit_idx])
Xfit = sc.transform(Xall[fit_idx])
Xeval = sc.transform(np.vstack([Xall[held_normal], Xall[anomaly_idx]]))
yeval = np.r_[np.ones(len(held_normal)), np.zeros(len(anomaly_idx))]


print(f"So mau train (lanh tinh): {len(fit_idx)}")
print(f"So mau test:              {len(Xeval)}")
print(f"Ti le lanh tinh trong test: {yeval.mean():.3f}")

2. Train one-class SVM.

Kernel RBF, nu = 0.1. Sau khi train, decision_function(x) trả về điểm g(x): dương là giống lành tính, âm là bất thường. Vẽ historgarm để minh họa, ta thấy được hai lớp chồng lấn ở vùng quanh 0, đây là vùng xác suất có ý nghĩa.

In [ ]:
svm = OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
svm.fit(Xfit)

g_fit  = svm.decision_function(Xfit)
g_eval = svm.decision_function(Xeval)
fmax = g_fit.max()   # luu lai de dung trong gamma scaling

# ve histogram
plt.figure(figsize=(7, 3))
plt.hist(g_eval[yeval == 1], bins=24, color='#2a6', alpha=0.6, label='lanh tinh')
plt.hist(g_eval[yeval == 0], bins=24, color='#c33', alpha=0.6, label='ac tinh')
plt.axvline(0, color='k', lw=0.7, ls='--')
plt.xlabel('g(x) = decision value')
plt.ylabel('so mau')
plt.legend()
plt.title('Phan bo decision value tren tap test')
plt.tight_layout()
plt.show()

3. Bốn phương pháp chuyển điểm số sang xác suất.

Trong bài báo để cập tới bốn phương pháp chuyển đổi số sang xác suất

Platt: fit sigmoid với nhãn giả từ OCSVM (nhưng theo bài báo thì phương pháp này không hiệu quả, có thể collapse thành bước 0/1)

Equidistant binning: chia khoảng [g_min, 0] và [0, g_max] đều nhau.

Density binning: chia theo quantile, đây là cách LIBSVM 3.3+ dùng mặc định.

New Gamma scaling: fit Gamma cho S = max(g_max - g, 0), scale sao cho g = 0 cho ra xác suất 0.5.

In [ ]:
def platt(g_train, g_query, pseudo):
    pseudo = pseudo.astype(float)
    Np = pseudo.sum()
    Nn = len(pseudo) - Np
    # target voi smoothing chong overfit
    t = np.where(pseudo > 0, (Np + 1) / (Np + 2), 1 / (Nn + 2))

    def J(ab):
        z = ab[0] * g_train + ab[1]
        return (t * z + np.log1p(np.exp(-z))).sum()

    out = minimize(J, [0.0, 0.0], method='Nelder-Mead', options={'xatol': 1e-5})
    A, B = out.x
    return expit(-(A * g_query + B))

# nhan gia: svm.predict tra +1 cho mau giong train, -1 cho bat thuong
pseudo = (svm.predict(Xfit) > 0).astype(int)

In [ ]:
# binning methods
EPS = 0.001
K = 5  # so thung moi ben g = 0

def _nearest_mark(marks, probs, q):
    # gan moi diem query vao mark gan nhat, lay xac suat tuong ung
    j = np.abs(q[:, None] - marks[None, :]).argmin(axis=1)
    return probs[j]

def equidistant(g_train, g_query):
    # Binning chia khoang deu nhau
    a, b = g_train.min(), g_train.max()
    left  = np.linspace(a, 0, K + 1)[:-1]
    right = np.linspace(0, b, K + 1)[1:]
    marks = np.r_[left, 0.0, right]
    ps = np.linspace(EPS, 1 - EPS, marks.size)
    return _nearest_mark(marks, ps, g_query)

def density_bin(g_train, g_query):
    # Binning chia theo quantile (LIBSVM default)
    qs = (np.arange(K) + 0.5) / K
    neg = g_train[g_train < 0]
    pos = g_train[g_train >= 0]
    nm = np.quantile(neg, qs) if len(neg) else np.zeros(K)
    pm = np.quantile(pos, qs) if len(pos) else np.zeros(K)
    marks = np.r_[nm, 0.0, pm]
    ps = np.linspace(EPS, 1 - EPS, marks.size)
    return _nearest_mark(marks, ps, g_query)

In [ ]:
# New Gamma scaling - phuong phap moi cua bai bao (cong thuc 40)
def gamma_scale(g_train, g_query):
    # buoc 1: tinh S = (fmax - g)+ tren tap train
    S = np.clip(fmax - g_train, 0, None)
    S = S[S > 0]

    # buoc 2: fit Gamma bang phuong phap moment
    # (Satterthwaite-Welch xap xi tong chi-binh-phuong)
    m, v = S.mean(), S.var()
    shape, scale = m * m / v, v / m

    # buoc 3: tinh survival function
    Sq = np.clip(fmax - g_query, 0, None)
    surv = 1 - gamma.cdf(Sq, shape, scale=scale)
    surv0 = 1 - gamma.cdf(fmax, shape, scale=scale)
    surv0 = max(surv0, 1e-9)  # tranh chia 0

    # buoc 4: scale theo cong thuc (40) sao cho g=0 -> p=0.5
    up = 0.5 + 0.5 * (surv - surv0) / max(1 - surv0, 1e-9)
    dn = 0.5 * surv / surv0
    return np.where(surv >= surv0, up, dn).clip(EPS, 1 - EPS)

4. Test thử.

Lấy ba bệnh nhân ngẫu nhiên rồi chạy bốn phương pháp, check kết quả

In [ ]:
# tinh xac suat theo ca 4 phuong phap
probs = {
    'platt':       platt(g_fit, g_eval, pseudo),
    'equi-bin':    equidistant(g_fit, g_eval),
    'dens-bin':    density_bin(g_fit, g_eval),
    'gamma-scale': gamma_scale(g_fit, g_eval),
}

# chon 3 benh nhan ngau nhien
trio = rng.choice(len(g_eval), size=3, replace=False)

header = f"{'row':>4}  {'g':>7}  {'y':>2}    " + "   ".join(f"{k:>11}" for k in probs)
print(header)
print("-" * len(header))
for r in trio:
    row = f"{r:>4}  {g_eval[r]:+.3f}  {int(yeval[r]):>2}    "
    row += "   ".join(f"{probs[k][r]:>11.3f}" for k in probs)
    print(row)

5. Reliability plot.

Chia tập test thành 10 thùng quantile theo xác suất dự đoán. Mỗi thùng lấy xác suất trung bình và tỉ lệ lành tính thực tế. Mô hình calibrated tốt sẽ nằm trên đường chéo y = x: mô hình nói 80% thì 80/100 ca như thế phải là lành tính thật.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot([0, 1], [0, 1], color='#999', lw=0.8, ls='--', label='hoan hao (y=x)')

for name, p in probs.items():
    # 10 thung quantile
    e = np.quantile(p, np.linspace(0, 1, 11))
    e[0] -= 1e-6
    e[-1] += 1e-6
    xs, ys = [], []
    for i in range(10):
        m = (p >= e[i]) & (p < e[i + 1])
        if m.sum() >= 2:
            xs.append(p[m].mean())
            ys.append(yeval[m].mean())
    ax.plot(xs, ys, '.-', label=name, lw=1.2, markersize=8)

ax.set_xlabel('xac suat du doan (mean trong thung)')
ax.set_ylabel('ti le lanh tinh thuc te')
ax.set_title('Reliability plot')
ax.legend(frameon=False, loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

6. Brier score.

Brier = MSE giữa xác suất dự đoán và nhãn thật (smaller is better)

In [ ]:
print(f"{'phuong phap':<14} Brier score")
print("-" * 30)
for name, p in probs.items():
    brier = np.mean((p - yeval) ** 2)
    print(f"  {name:<12} {brier:.4f}")